# IFEval과 한국어 전용 응답의 전후 비교

기존 조건과 system 지시·assistant prefix를 함께 적용한 조건을 같은 모델·질문으로 평가한다. IFEval 채점은 DeepEval을 사용하고, 한국어 평가는 한글 전용 응답 여부를 검사한다. 개선 효과는 실행 결과로 판단하며, 점수 상승을 전제하지 않는다.

이 노트북이 있는 폴더에서 실행한다. `.env`에 `MISTRAL_API_STUDIO`를 설정한다. 기본값은 모델 4개, 평가별 앞 19문항으로 총 304회 생성 요청이다. 수정 전에는 4개 모델, prefix 조건에는 지원 모델 3개, system 지시만 적용하는 조건에는 Mistral Large 4를 평가한다. `N_PROBLEMS = None`이면 IFEval 541문항과 한국어 20문항을 사용한다.

응답과 판정은 `results/`에 문항마다 저장한다. 같은 평가 셀을 다시 실행하면 이미 저장한 모델·문항 조합은 건너뛰고 나머지를 새로 요청한다. 조건을 바꾼 실험은 결과 파일을 다른 이름으로 저장한다. 저장된 결과를 다시 볼 때는 생성 셀을 건너뛰고 시각화 셀만 실행한다.

In [ ]:
import json
import os
from pathlib import Path
import unicodedata

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display
from dotenv import load_dotenv
from mistralai.client import Mistral

os.environ["DEEPEVAL_TELEMETRY_OPT_OUT"] = "1"
from deepeval.benchmarks import IFEval

load_dotenv()

MODELS = [
    "mistral-small-2603",
    "mistral-large-4",
    "zai-glm-5-3",
    "deepseek-v4-1-flash-1017",
]
PREFIX_MODELS = [name for name in MODELS if name != "mistral-large-4"]
SYSTEM_MODELS = [name for name in MODELS if name == "mistral-large-4"]
N_PROBLEMS = 19  # None이면 모든 문항을 사용한다.
RESULTS = Path("results")
RESULTS.mkdir(exist_ok=True)

KOREAN_INSTRUCTION = (
    "한국어로만 답하세요. 외국어 단어나 영문 약어는 한글로 풀어 쓰세요. "
    "숫자와 문장부호는 사용할 수 있습니다."
)
IFEVAL_SYSTEM = (
    "Follow every instruction in the user's request, including language, "
    "format, length, and wording. Continue after the supplied prefix with "
    "only the requested answer. Do not add an introduction or commentary."
)
IFEVAL_PREFIX = "Here is the requested answer:\n"
KOREAN_SYSTEM = (
    KOREAN_INSTRUCTION
    + " 괄호 안에도 외국어 원문을 병기하지 마세요. "
    "외국 이름은 한글로 표기하세요. 주어진 시작 문구 다음에 답변 본문만 작성하세요."
)
KOREAN_PREFIX = "한국어 답변:\n"

## 모델 호출과 저장

Mistral 공식 prefix 예제처럼 마지막 assistant 메시지에 `prefix=True`를 지정한다. 수정 후에는 system 지시도 함께 추가한다.

IFEval prefix는 `Here is the requested answer:\n`, 한국어 prefix는 `한국어 답변:\n`이다. 미리 제공한 prefix는 응답 맨 앞에 있을 때만 제거한다. 제거한 본문을 최종 응답으로 저장하고 채점한다. 이 처리로 고정 안내 문구가 JSON·첫 문장 지시를 깨뜨리거나, 주입한 한글만으로 한국어 검사를 통과하는 것을 방지한다. 모델이 생성한 외국어·문장부호·공백은 별도로 삭제하지 않는다.

이 비교는 system 지시, prefix, prefix 제거를 합친 방식의 효과를 측정한다. prefix만의 효과를 분리한 실험이나 Google 원본 채점기 결과가 아니다.

참고: [Mistral prefix 예제](https://docs.mistral.ai/resources/cookbooks/mistral-prompting-prefix_use_cases).

In [3]:
class MistralModel:
    def __init__(self, client, name, system="", prefix=""):
        self.client = client
        self.name = name
        self.system = system
        self.prefix = prefix

    def generate(self, prompt, schema=None):
        messages = []
        if self.system:
            messages.append({"role": "system", "content": self.system})
        messages.append({"role": "user", "content": prompt})
        if self.prefix:
            messages.append({
                "role": "assistant",
                "content": self.prefix,
                "prefix": True,
            })

        response = self.client.chat.complete(model=self.name, messages=messages)
        content = response.choices[0].message.content
        if isinstance(content, str):
            text = content
        else:
            text = "".join(
                chunk.text for chunk in (content or []) if chunk.type == "text"
            )

        # 응답 맨 앞에 있는 고정 prefix만 제거한다.
        text = text.removeprefix(self.prefix)
        if schema is not None:
            return schema(answer=text)
        return text


def load_results(filename):
    path = RESULTS / filename
    if not path.exists():
        return []
    return json.loads(path.read_text(encoding="utf-8"))


def save_results(filename, rows):
    (RESULTS / filename).write_text(
        json.dumps(rows, ensure_ascii=False, indent=2), encoding="utf-8"
    )


## IFEval 데이터와 평가 함수

DeepEval이 불러온 원본 문항을 두 조건에서 그대로 사용한다. 응답의 모든 지시 통과 여부와 지시별 판정을 저장한다. IFEval이 요구하는 언어는 문항에 따라 다르므로 한국어 지시를 추가하지 않는다.

점수는 설치된 DeepEval의 채점 구현을 따른다. 일부 원본 지시를 충분히 검사하지 않는 기존 문제는 이 실험에서 수정하지 않는다.

In [4]:
benchmark = IFEval()
english = benchmark.load_benchmark_dataset()[:N_PROBLEMS]


def evaluate_ifeval(filename, models, system="", prefix=""):
    rows = load_results(filename)
    done = {(row["model"], row["prompt"]) for row in rows}
    with Mistral(api_key=os.environ["MISTRAL_API_STUDIO"]) as client:
        for name in models:
            print(f"{filename}: {name}")
            model = MistralModel(client, name, system, prefix)
            for example in english:
                if (name, example.input) in done:
                    continue
                response, passed, instruction_scores = benchmark.predict(model, example)
                rows.append({
                    "model": name,
                    "prompt": example.input,
                    "response": response,
                    "passed": passed,
                    "instruction_scores": instruction_scores,
                })
                save_results(filename, rows)


d:\workspace\codes\research\ai-cookbook\evaluation\korean-language-mixing\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 한국어 데이터와 평가 함수

`data/prompts.json`의 질문 뒤에 같은 `KOREAN_INSTRUCTION`을 붙여 두 조건에 보낸다. 수정 후에는 system 지시와 한국어 prefix를 추가한다.

한글 문자가 최소 하나 있어야 하며, 다른 언어의 문자가 하나라도 있으면 실패다. 영문 약어와 고유명사도 한글로 써야 한다. 숫자·공백·문장부호·기호는 언어 문자로 세지 않는다. 빈 응답이나 숫자만 있는 응답은 실패한다. 이 검사는 답변의 정확성이나 자연스러움을 평가하지 않는다.

In [ ]:
korean = json.loads(
    Path("data/prompts.json").read_text(encoding="utf-8")
)[:N_PROBLEMS]


def evaluate_korean(filename, models, system="", prefix=""):
    rows = load_results(filename)
    done = {(row["model"], row["prompt"]) for row in rows}
    with Mistral(api_key=os.environ["MISTRAL_API_STUDIO"]) as client:
        for name in models:
            print(f"{filename}: {name}")
            model = MistralModel(client, name, system, prefix)
            for question in korean:
                if (name, question) in done:
                    continue
                prompt = f"{question}\n\n{KOREAN_INSTRUCTION}"
                response = model.generate(prompt)
                rows.append({
                    "model": name,
                    "prompt": question,
                    "response": response,
                    "passed": is_korean_only(response),
                })
                save_results(filename, rows)


## 수정 전 응답 생성과 저장

다음 두 셀은 실제 모델을 호출한다. 한국어만 평가하려면 IFEval 데이터·함수 셀과 IFEval 생성 셀을 건너뛴다.

In [ ]:
evaluate_ifeval("ifeval_before.json", MODELS)

In [ ]:
evaluate_korean("korean_before.json", MODELS)

## 저장된 수정 전 결과 시각화

이 셀은 모델을 호출하지 않는다. 파일의 문항 수·통과 수·통과율을 표시하고 그래프를 PNG로 저장한다. 한국어만 실행했다면 `show_before("korean")`만 호출한다.

In [ ]:
def show_before(name):
    rows = pd.read_json(RESULTS / f"{name}_before.json")
    scores = rows.groupby("model", sort=False).agg(
        questions=("passed", "size"), passed=("passed", "sum")
    )
    scores["pass_rate"] = 100 * scores["passed"] / scores["questions"]
    display(scores)

    fig, ax = plt.subplots(figsize=(10, 4))
    ax = scores["pass_rate"].plot.barh(
        ax=ax, xlim=(0, 100), color="#666666", title=f"{name}: before"
    )
    ax.set_xlabel("Pass rate (%)")
    ax.set_ylabel("")
    ax.set_yticklabels([
        f"{model} (n={count})" for model, count in scores["questions"].items()
    ])
    ax.invert_yaxis()
    ax.figure.tight_layout()
    # Matplotlib의 기본 Software 정보를 기록하지 않는다.
    ax.figure.savefig(
        RESULTS / f"{name}_before.png", metadata={"Software": None}
    )
    plt.show()
    plt.close(fig)


show_before("ifeval")
show_before("korean")

## 수정 후 응답 생성과 저장: system 지시와 prefix

PREFIX_MODELS의 3개 모델에 같은 문항으로 system 지시·prefix를 추가한 조건을 실행한다. 두 생성 셀은 수정 전 파일과 별개 파일에 저장한다.

In [ ]:
evaluate_ifeval("ifeval_after.json", PREFIX_MODELS, IFEVAL_SYSTEM, IFEVAL_PREFIX)

In [ ]:
evaluate_korean("korean_after.json", PREFIX_MODELS, KOREAN_SYSTEM, KOREAN_PREFIX)

## Prefix 미지원 모델: system 지시만 적용

Mistral Large 4는 현재 API에서 `Assistant message prefix is not enabled for this model` 오류를 반환한다. 이 모델에는 동일한 system 지시만 추가하고 assistant prefix는 보내지 않는다.

응답과 판정은 `ifeval_system_only.json`, `korean_system_only.json`에 별도로 저장한다. 이 결과를 prefix 효과로 해석하지 않는다. `SYSTEM_MODELS`는 이 조건을 실행할 모델 목록이다.

In [ ]:
evaluate_ifeval("ifeval_system_only.json", SYSTEM_MODELS, IFEVAL_SYSTEM)

In [ ]:
evaluate_korean("korean_system_only.json", SYSTEM_MODELS, KOREAN_SYSTEM)

## 저장된 전후 결과 비교

같은 모델·질문을 기준으로 전후 결과를 연결한다. 중간에 실행이 멈춘 경우 양쪽에 결과가 있는 문항만 비교하며, `questions` 열에서 비교 문항 수를 확인한다. `change_pp`는 해당 수정 조건의 통과율에서 수정 전 통과율을 뺀 퍼센트포인트 차이다. Prefix와 system 지시만 적용한 조건은 서로 다른 그래프로 표시한다.

prefix 비교는 `ifeval_comparison.png`와 `korean_comparison.png`, system 지시만 적용한 비교는 `ifeval_system_comparison.png`와 `korean_system_comparison.png`에 저장한다. 한국어만 실행했다면 `compare_results("korean")`만 호출한다. 생성 코드나 API 키 없이도 필요한 라이브러리와 `RESULTS` 경로를 설정해 저장된 결과를 다시 비교할 수 있다.

In [ ]:
def compare_results(name, condition="after"):
    before = pd.read_json(RESULTS / f"{name}_before.json")
    after = pd.read_json(RESULTS / f"{name}_{condition}.json")
    label = "System + prefix" if condition == "after" else "System only"
    suffix = "comparison" if condition == "after" else "system_comparison"
    paired = before.merge(
        after, on=["model", "prompt"], suffixes=("_before", "_after")
    )

    grouped = paired.groupby("model", sort=False)
    scores = grouped[["passed_before", "passed_after"]].mean().mul(100)
    scores.columns = ["Before", label]
    scores["change_pp"] = scores[label] - scores["Before"]
    scores["questions"] = grouped.size()
    print(f"{name} / {label}: 같은 모델·질문만 비교")
    display(scores.round(1))

    fig, ax = plt.subplots(figsize=(10, 4))
    ax = scores[["Before", label]].plot.barh(
        ax=ax, xlim=(0, 100),
        color=["#666666", "#176b93"], title=f"{name}: before / {label}",
    )
    ax.set_xlabel("Pass rate (%)")
    ax.set_ylabel("")
    ax.set_yticklabels([
        f"{model} (n={count})" for model, count in scores["questions"].items()
    ])
    ax.invert_yaxis()
    ax.legend(loc="upper left", bbox_to_anchor=(1, 1), frameon=False)
    ax.figure.tight_layout()
    # Matplotlib의 기본 Software 정보를 기록하지 않는다.
    ax.figure.savefig(
        RESULTS / f"{name}_{suffix}.png", metadata={"Software": None}
    )
    plt.show()
    plt.close(fig)
    return paired


ifeval_comparison = compare_results("ifeval")
korean_comparison = compare_results("korean")
ifeval_system_comparison = compare_results("ifeval", "system_only")
korean_system_comparison = compare_results("korean", "system_only")

## 판정이 바뀐 응답 확인

실패에서 통과로 바뀐 경우와 통과에서 실패로 바뀐 경우를 함께 확인한다. 모든 응답은 JSON 파일에 남아 있다. 한글만 사용하는 오답도 통과할 수 있으므로 내용도 읽어 본다.

기본 한국어 19문항 중 15문항에는 외국어가 포함되어 있다. 적은 문항을 한 번씩 실행한 결과를 일반적인 한국어 대화 전체의 혼용률이나 통계적으로 확정된 개선으로 해석하지 않는다.

In [ ]:
columns = [
    "model", "prompt", "response_before", "response_after",
    "passed_before", "passed_after",
]
for name, paired in [
    ("IFEval / prefix", ifeval_comparison),
    ("Korean / prefix", korean_comparison),
    ("IFEval / system only", ifeval_system_comparison),
    ("Korean / system only", korean_system_comparison),
]:
    print(f"{name}: 판정이 바뀐 문항")
    display(paired.loc[
        paired["passed_before"] != paired["passed_after"], columns
    ])